In [4]:
import os
import glob
from goatools.base import download_go_basic_obo
from goatools.obo_parser import GODag
from goatools.anno.gaf_reader import GafReader
from goatools.goea.go_enrichment_ns import GOEnrichmentStudyNS
import csv
import glob


In [ ]:

obo_fname = download_go_basic_obo()
godag = GODag(obo_fname)

gaf_path = "/home/cassandre/stage/Cassandre/Code/PHASE_2/goatoolsfiles/goa_mouse.gaf"
mouse_map = GafReader(gaf_path, godag=godag) 

ns2assoc = mouse_map.get_ns2assc()

  EXISTS: go-basic.obo
go-basic.obo: fmt(1.2) rel(2026-03-25) 41,853 Terms
HMS:0:00:10.555705 656,063 annotations READ: /home/cassandre/stage/Cassandre/Code/PHASE_2/goatoolsfiles/goa_mouse.gaf 


In [7]:
bg_file = "/home/cassandre/stage/Cassandre/ortho/comparison_results/human_mus/background_mouse_h_protT5_oto.txt"

with open(bg_file, 'r') as f:
    background_ids = [line.strip() for line in f if line.strip()]

gaf_path = "/home/cassandre/stage/Cassandre/Code/PHASE_2/goatoolsfiles/goa_mouse.gaf"
mouse_map = GafReader(gaf_path, godag=godag) 

# 3. Get the associations (Mappings of protein ID -> GO terms)
ns2assoc = mouse_map.get_ns2assc()

HMS:0:00:11.039775 656,063 annotations READ: /home/cassandre/stage/Cassandre/Code/PHASE_2/goatoolsfiles/goa_mouse.gaf 


In [8]:
goeaobj = GOEnrichmentStudyNS(
    background_ids, 
    ns2assoc, 
    godag,
    propagate_counts = False,
    alpha = 0.05, 
    methods = ['fdr_bh']
)



Load BP Ontology Enrichment Analysis ...
 90% 14,819 of 16,508 population items found in association

Load CC Ontology Enrichment Analysis ...
 95% 15,621 of 16,508 population items found in association

Load MF Ontology Enrichment Analysis ...
 85% 14,030 of 16,508 population items found in association


In [9]:
input_folder = "/home/cassandre/stage/Cassandre/ortho/comparison_results/human_mus"
output_csv = "/home/cassandre/stage/Cassandre/ortho/comparison_results/human_mus/GO_summary_humanmouse.csv"

master_results = []

search_path = os.path.join(input_folder, "*_fail.txt")
for file_path in glob.glob(search_path):
    file_name = os.path.basename(file_path)
    print(f"Processing: {file_name}...")
    
    parts = file_name.replace("_fail.txt", "").split("_")
    model = parts[0] if len(parts) > 0 else "Unknown"
    ortho = parts[1] if len(parts) > 1 else "Unknown"  #get the part of my file name into variables cuz im smaaaart and name them right
    organism = parts[2] if len(parts) > 2 else "Unknown"

    with open(file_path, 'r') as f:
        study_ids = [line.strip() for line in f if line.strip()] 
    
    if not study_ids:
        continue

    results_all = goeaobj.run_study(study_ids)
    
    # 3. Filter using the Corrected p-value (FDR / Benjamini-Hochberg)
    for res in results_all:
        # We use p_fdr_bh because it's the statistical standard for GOEA
        if res.p_fdr_bh < 0.05 and res.goterm.namespace == 'biological_process':
            
            # Calculating percentages to determine 'strength' of failure
            study_perc = res.study_count / res.study_n
            pop_perc = res.pop_count / res.pop_n
            fold = study_perc / pop_perc if pop_perc > 0 else 0
            
            direction = "+" if study_perc > pop_perc else "-"
            
            # Store everything for the CSV
            master_results.append({
                'Model': model,
                'OrthoType': ortho,
                'Organism': organism,
                'GO_ID': res.goterm.id,
                'Term_Name': res.goterm.name,
                'p_val_raw': res.p_uncorrected,   
                'p_val_corr': res.p_fdr_bh,      
                'Fold_Enrichment': round(fold, 2),
                'Direction': direction,
                'Count': res.study_count
            })

# save to csv file to the output variable
if master_results:
    keys = master_results[0].keys()
    with open(output_csv, 'w', newline='') as f:
        dict_writer = csv.DictWriter(f, fieldnames=keys)
        dict_writer.writeheader() 
        dict_writer.writerows(master_results)
    print(f"saved to {output_csv}")
else:
    print("No GO terms that respects the condition.")

Processing: ESM600M_OtM_mouse_fail.txt...

Runing BP Ontology Analysis: current study set of 247 IDs.
  0%      0 of      0 study items found in association
  0%      0 of    247 study items found in population(16508)

Runing CC Ontology Analysis: current study set of 247 IDs.
  0%      0 of      0 study items found in association
  0%      0 of    247 study items found in population(16508)

Runing MF Ontology Analysis: current study set of 247 IDs.
  0%      0 of      0 study items found in association
  0%      0 of    247 study items found in population(16508)
Processing: ESM600M_MtO_human_fail.txt...

Runing BP Ontology Analysis: current study set of 230 IDs.
  0%      0 of      0 study items found in association
  0%      0 of    230 study items found in population(16508)

Runing CC Ontology Analysis: current study set of 230 IDs.
  0%      0 of      0 study items found in association
  0%      0 of    230 study items found in population(16508)

Runing MF Ontology Analysis: curren

use corrected p-value instead = fdr 